In [0]:
schema_location = "abfss://bronze@adlsearthquake.dfs.core.windows.net/schema"
checkpoint_location = "abfss://bronze@adlsearthquake.dfs.core.windows.net/checkpoint"

df = spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", "json") \
        .option("cloudFiles.schemaLocation", schema_location) \
        .load("abfss://raw@adlsearthquake.dfs.core.windows.net/earthquakes/")

query = df.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_location) \
    .option("mergeSchema", "true") \
    .option("path","abfss://bronze@adlsearthquake.dfs.core.windows.net/earthquake_data") \
    .trigger(availableNow=True) \
    .start()

query.awaitTermination()